# Task 2: Learning to Rank on Internet Mathematics 2009 Data

In [ ]:
def convert_to_letor_dense(input_path, output_path, n_features=245):
    with open(input_path, 'r') as fin, open(output_path, 'w') as fout:
        for doc_idx, line in enumerate(fin):
            line = line.strip()
            if not line:
                continue
            # split off comment
            if '#' in line:
                main, comment = line.split('#', 1)
                query_id = comment.strip()
            else:
                main = line
                query_id = '0'
            parts = main.strip().split()
            label = round(float(parts[0]))   # round float to integer
            label = max(0, min(4, label))    # clamp to [0,4]

            # parse sparse features into dict
            feat_dict = {}
            for fv in parts[1:]:
                fid, val = fv.split(':')
                feat_dict[int(fid)] = val

            # build dense feature string 1..n_features
            dense_feats = ' '.join(
                f'{i}:{feat_dict.get(i, "0.000000")}' for i in range(1, n_features + 1)
            )
            fout.write(f'{label} qid:{query_id} {dense_feats} # {doc_idx}\n')

convert_to_letor_dense(
    'imat2009_new_split/imat2009_train_new.txt',
    'imat2009_train_letor.txt'
)
convert_to_letor_dense(
    'imat2009_new_split/imat2009_test_new.txt',
    'imat2009_test_letor.txt'
)
print('Conversion done.')

with open('imat2009_train_letor.txt') as f:
    line = f.readline()
    parts = line.split()
    print('Label:', parts[0])
    print('QID:  ', parts[1])
    print('First 5 features:', ' '.join(parts[2:7]))
    print('Last 5 features: ', ' '.join(parts[-6:-1]))
    print('Total fields:', len(parts))

Conversion done.
Label: 1
QID:   qid:3382
First 5 features: 1:0.000023 2:0.000000 3:0.000000 4:0.000000 5:0.000000
Last 5 features:  242:0.000000 243:0.000023 244:1.000000 245:0.000023 #
Total fields: 249


In [2]:
# Cell 2: Describe the data
import numpy as np
import pandas as pd
from collections import Counter

def describe_letor(path, name):
    labels, queries, feat_counts = [], [], []
    query_docs = Counter()
    with open(path) as f:
        for line in f:
            line = line.strip()
            if not line: continue
            main = line.split('#')[0].strip().split()
            label = int(main[0])
            qid = main[1].split(':')[1]
            feats = main[2:]
            labels.append(label)
            queries.append(qid)
            query_docs[qid] += 1
            feat_counts.append(len(feats))

    labels = np.array(labels)
    docs_per_q = list(query_docs.values())
    print(f'=== {name} ===')
    print(f'Total documents:      {len(labels)}')
    print(f'Unique queries:       {len(query_docs)}')
    print(f'Avg docs/query:       {np.mean(docs_per_q):.1f}')
    print(f'Min/Max docs/query:   {min(docs_per_q)} / {max(docs_per_q)}')
    print(f'Features (avg/line):  {np.mean(feat_counts):.1f} (sparse out of 245)')
    print(f'Label distribution:')
    for lv in range(5):
        count = np.sum(labels == lv)
        pct = 100 * count / len(labels)
        print(f'  {lv}: {count:6d} ({pct:.1f}%)')
    print()

describe_letor('imat2009_train_letor.txt', 'Train')
describe_letor('imat2009_test_letor.txt',  'Test')

=== Train ===
Total documents:      77714
Unique queries:       7300
Avg docs/query:       10.6
Min/Max docs/query:   2 / 461
Features (avg/line):  245.0 (sparse out of 245)
Label distribution:
  0:  28128 (36.2%)
  1:  20647 (26.6%)
  2:  26138 (33.6%)
  3:   1823 (2.3%)
  4:    978 (1.3%)



=== Test ===
Total documents:      19576
Unique queries:       1824
Avg docs/query:       10.7
Min/Max docs/query:   2 / 434
Features (avg/line):  245.0 (sparse out of 245)
Label distribution:
  0:   6975 (35.6%)
  1:   5219 (26.7%)
  2:   6383 (32.6%)
  3:    835 (4.3%)
  4:    164 (0.8%)



In [3]:
# Cell 3: Load LETOR data into CatBoost Pools
from catboost import CatBoostRanker, Pool
import numpy as np
from sklearn.datasets import load_svmlight_file

def load_letor(path):
    X, y, qids = [], [], []
    with open(path) as f:
        for line in f:
            line = line.strip()
            if not line: continue
            main = line.split('#')[0].strip().split()
            label = int(main[0])
            qid = main[1].split(':')[1]
            feats = [float(fv.split(':')[1]) for fv in main[2:]]
            X.append(feats)
            y.append(label)
            qids.append(qid)
    return np.array(X, dtype=np.float32), np.array(y), np.array(qids)

print('Loading train...')
X_train, y_train, qids_train = load_letor('imat2009_train_letor.txt')
print('Loading test...')
X_test, y_test, qids_test = load_letor('imat2009_test_letor.txt')

train_pool = Pool(data=X_train, label=y_train, group_id=qids_train)
test_pool  = Pool(data=X_test,  label=y_test,  group_id=qids_test)

print(f'Train: {X_train.shape}, Test: {X_test.shape}')
print('Pools created.')

Loading train...


Loading test...


Train: (77714, 245), Test: (19576, 245)
Pools created.


In [4]:
# Cell 4: Method 1 - YetiRank (approximates NDCG)
model_yetirank = CatBoostRanker(
    loss_function='YetiRank',
    eval_metric='NDCG',
    iterations=500,
    learning_rate=0.3,
    random_seed=0,
    verbose=100,
)
model_yetirank.fit(train_pool, eval_set=test_pool)
print('YetiRank done.')

0:	test: 0.8264357	best: 0.8264357 (0)	total: 93.3ms	remaining: 46.5s


100:	test: 0.8878978	best: 0.8889640 (52)	total: 2.83s	remaining: 11.2s


200:	test: 0.8873611	best: 0.8889640 (52)	total: 5.51s	remaining: 8.2s


300:	test: 0.8867848	best: 0.8889640 (52)	total: 8.22s	remaining: 5.43s


400:	test: 0.8870279	best: 0.8889640 (52)	total: 11s	remaining: 2.71s


499:	test: 0.8870673	best: 0.8889640 (52)	total: 13.7s	remaining: 0us

bestTest = 0.8889640197
bestIteration = 52

Shrink model to first 53 iterations.
YetiRank done.


In [5]:
# Cell 5: Method 2 - LambdaMart (directly optimizes NDCG)
model_lambdamart = CatBoostRanker(
    loss_function='LambdaMart:metric=NDCG',
    eval_metric='NDCG',
    iterations=500,
    learning_rate=0.1,
    random_seed=0,
    verbose=100,
)
model_lambdamart.fit(train_pool, eval_set=test_pool)
print('LambdaMart done.')

0:	test: 0.8196557	best: 0.8196557 (0)	total: 14.1ms	remaining: 7.03s


100:	test: 0.8865400	best: 0.8867996 (91)	total: 1.32s	remaining: 5.2s


200:	test: 0.8897032	best: 0.8902527 (185)	total: 2.58s	remaining: 3.84s


300:	test: 0.8911426	best: 0.8918533 (267)	total: 3.81s	remaining: 2.52s


400:	test: 0.8907979	best: 0.8918533 (267)	total: 5.03s	remaining: 1.24s


499:	test: 0.8921470	best: 0.8924130 (453)	total: 6.23s	remaining: 0us

bestTest = 0.8924129968
bestIteration = 453

Shrink model to first 454 iterations.
LambdaMart done.


In [6]:
# Cell 6: Report results
import pandas as pd

results = []
for name, model in [('YetiRank', model_yetirank), ('LambdaMart (NDCG)', model_lambdamart)]:
    metrics = model.eval_metrics(test_pool, ['NDCG', 'AverageGain:top=10'])
    ndcg_key     = [k for k in metrics if 'NDCG' in k][0]
    avg_gain_key = [k for k in metrics if 'AverageGain' in k][0]
    results.append({
        'Method':         name,
        'NDCG':           round(metrics[ndcg_key][-1], 4),
        'AverageGain@10': round(metrics[avg_gain_key][-1], 4),
        'Best Iteration': model.get_best_iteration(),
    })

df = pd.DataFrame(results)
print(df.to_string(index=False))

           Method   NDCG  AverageGain@10  Best Iteration
         YetiRank 0.8890          0.9655              52
LambdaMart (NDCG) 0.8924          0.9653             453
